# Copy and deploy app.py and data to HISE

In [1]:
import glob
import hisepy
import os
import shutil
import tarfile
# import semantic_version
import hashlib
import pandas as pd

In [2]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Set version
0.0.1 <- bugfix  
0.1.0 <- new features  
1.0.0 <- breaks compatibility with old inputs

In [3]:
current_ver = "0.2.4"

## Retrieve input files

In [4]:
file_uuids = [
    "8aef5a04-a1b0-4117-af65-4bc158494900", # DEGs and original GSEA
    "a519a844-93f1-497d-89f1-01fed5e074df", # GSEA with P-value and Wald ranking
    "e9152a6e-f704-4eef-b1fd-01e2b536bad3"  # GSVA results
]

In [5]:
files_tar = hisepy.cache_files(file_uuids)

2026-06-26 12:03:04,732 INFO [hisepy.logging:185] logging 385 139839699408704 Calling cache_files
2026-06-26 12:03:17,027 INFO [hisepy.logging:228] logging 385 139839699408704 Finished cache_files successfully (time_elapsed=9.301s)


In [6]:
with tarfile.open(files_tar[0]) as tf:
    tf.extractall()

/tmp/ipykernel_385/1730903026.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall()


In [7]:
shutil.rmtree('deg_pipeline_pbmc/results/gsea')
shutil.rmtree('deg_pipeline_bmmc/results/gsea')

In [8]:
for f in files_tar[1:]:
    with tarfile.open(f) as tf:
        tf.extractall()

/tmp/ipykernel_385/1552163990.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall()


Hand-edit config file.

## Stage files to /home/jupyter/\<app>

In [9]:
target_path = '/home/workspace/NDMM-PBMC-{c}/'.format(c = current_ver)

app_subdirs = ['assets/', 'callbacks/', 'components/', 'config/', 'deg_pipeline_pbmc/', 'libs/', 'ui/']
app_file = 'app.py'
hero_file = 'hero_image.png'
version_file = 'version'
reqs_file = 'requirements.txt'

In [10]:
target_subdirs = [target_path + d for d in app_subdirs]

In [11]:
os.makedirs(target_path)
shutil.copy(app_file, target_path + app_file)
shutil.copy(hero_file, target_path + hero_file)
shutil.copy(version_file, target_path + version_file)
shutil.copy(reqs_file, target_path + reqs_file)

additional_files = []
for app_subdir in app_subdirs:
    target_subdir = target_path + app_subdir
    
    shutil.copytree(app_subdir, target_subdir)

    subdir_files = glob.glob(target_subdir + '*', recursive=True)
    
    additional_files = additional_files + subdir_files

In [12]:
additional_files

['/home/workspace/NDMM-PBMC-0.2.4/assets/deploy',
 '/home/workspace/NDMM-PBMC-0.2.4/assets/css',
 '/home/workspace/NDMM-PBMC-0.2.4/assets/images',
 '/home/workspace/NDMM-PBMC-0.2.4/assets/markdown',
 '/home/workspace/NDMM-PBMC-0.2.4/callbacks/create_deg_callbacks.py',
 '/home/workspace/NDMM-PBMC-0.2.4/components/create_deg_components.py',
 '/home/workspace/NDMM-PBMC-0.2.4/config/mm-pbmc.json',
 '/home/workspace/NDMM-PBMC-0.2.4/config/parse-celltypist.json',
 '/home/workspace/NDMM-PBMC-0.2.4/config/reflex.json',
 '/home/workspace/NDMM-PBMC-0.2.4/deg_pipeline_pbmc/results',
 '/home/workspace/NDMM-PBMC-0.2.4/libs/allen_dash_modules',
 '/home/workspace/NDMM-PBMC-0.2.4/ui/create_deg_layout.py']

In [13]:
target_path + app_file

'/home/workspace/NDMM-PBMC-0.2.4/app.py'

In [14]:
target_path + hero_file

'/home/workspace/NDMM-PBMC-0.2.4/hero_image.png'

In [15]:
target_path + version_file

'/home/workspace/NDMM-PBMC-0.2.4/version'

In [16]:
target_path + reqs_file

'/home/workspace/NDMM-PBMC-0.2.4/requirements.txt'

In [17]:
target_subdirs

['/home/workspace/NDMM-PBMC-0.2.4/assets/',
 '/home/workspace/NDMM-PBMC-0.2.4/callbacks/',
 '/home/workspace/NDMM-PBMC-0.2.4/components/',
 '/home/workspace/NDMM-PBMC-0.2.4/config/',
 '/home/workspace/NDMM-PBMC-0.2.4/deg_pipeline_pbmc/',
 '/home/workspace/NDMM-PBMC-0.2.4/libs/',
 '/home/workspace/NDMM-PBMC-0.2.4/ui/']

## Deploy to HISE

In [18]:
import session_info
session_info.show()

In [19]:
hisepy.save_dash_app(
    app_filepath = target_path + app_file,
    additional_files = [target_path + version_file], 
    input_file_ids = file_uuids,
    requirements = target_path + reqs_file,
    study_space_id = 'a7a0ad59-ce58-43c1-a702-8482724eb678', # deploy to MM project page
    title = f'Multiple Myeloma DEG - Peripheral Blood ({current_ver})',
    description = f'Multiple Myeloma DEG - Peripheral Blood ({current_ver}) with up-to-date links.',
    image = hero_file,
    additional_dirs = target_subdirs,
    do_conda_build_check=False
)

2026-06-26 12:03:22,075 INFO [hisepy.logging:185] logging 385 139839699408704 Calling save_dash_app
2026-06-26 12:03:24,128 INFO [hisepy.logging:686] upload 385 139839699408704 Created temporary directory for Dash app build: /home/workspace/a0weg4p4
2026-06-26 12:03:32,922 INFO [hisepy.logging:713] upload 385 139839699408704 Dash app packaged at: /home/workspace/a0weg4p4/dash_app.tar.gz
2026-06-26 12:03:32,923 INFO [hisepy.logging:142] upload 385 139839699408704 Uploading hero image...
2026-06-26 12:03:32,923 INFO [hisepy.logging:185] logging 385 139839699408704 Calling save_static_image
2026-06-26 12:03:39,248 INFO [hisepy.logging:228] logging 385 139839699408704 Finished save_static_image successfully (time_elapsed=3.488s)
2026-06-26 12:03:39,249 INFO [hisepy.logging:150] upload 385 139839699408704 Uploading Dash app bundle and dependencies...
2026-06-26 12:03:39,250 INFO [hisepy.logging:185] logging 385 139839699408704 Calling upload_files_internal


Please provide input of comma separated sample ids or sample kit guids for the files being uploaded. If you do not have any samples, press enter:  


Cannot determine the current notebook.
1) /home/workspace/ndmm-gsva/apps/ndmm-sc-deg-pbmc/deploy_pbmc.ipynb
2) /home/workspace/ndmm-gsva/apps/ndmm-sc-deg-bmmc/deploy_bmmc.ipynb
3) /home/workspace/ndmm-gsva/datasets/multiple_myeloma/13-Python_gsva_parquet_format.ipynb
Please select (1-3) 


 1


2026-06-26 12:08:22,524 INFO [hisepy.logging:228] logging 385 139839699408704 Finished upload_files_internal successfully (time_elapsed=278.369s)
2026-06-26 12:08:22,525 INFO [hisepy.logging:180] upload 385 139839699408704 Creating Dash workflow: https://allenimmunology.org/ide-nextgen/visualization/dash/workflow/e09b2c06-bd23-4958-b7ad-bd59af2a4f2a
2026-06-26 12:08:24,900 INFO [hisepy.logging:718] upload 385 139839699408704 Dash app successfully uploaded and deployed.
2026-06-26 12:08:28,754 INFO [hisepy.logging:228] logging 385 139839699408704 Finished save_dash_app successfully (time_elapsed=302.826s)


{'Message': 'Dash app workflow initiated',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'e09b2c06-bd23-4958-b7ad-bd59af2a4f2a',
 'ProcessId': '00000000-0000-0000-0000-000000000000',
 'WorkflowId': 'd2f99d38-dbae-4740-a73a-00032951d20a'}